In [1]:
from pathlib import Path
import re
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Pasta onde estão as imagens
pasta_dataset = Path("dataset")

arquivos_dataset = []
for ext in ("*.png", "*.jpg", "*.jpeg"):
    arquivos_dataset.extend(sorted(pasta_dataset.glob(ext)))


def extrair_rotulo(nome_arquivo: str):
    """
    Extrai o dígito no início do nome.

    Exemplos:
        8_amostra_003 -> 8
        7_amostra_001 -> 7
        3_teste_10    -> 3
    """
    m = re.match(r"^(\d)_", nome_arquivo)

    if m:
        return int(m.group(1))

    return None


X = []
y = []
ignorados = []

for arq in arquivos_dataset:

    # Pega o rótulo pelo nome do arquivo
    rotulo = extrair_rotulo(arq.stem)

    if rotulo is None:
        ignorados.append(arq.name)
        continue

    # Abre a imagem, transforma em escala de cinza e redimensiona
    img = Image.open(arq).convert("L").resize((8, 8))

    # Converte para array
    arr = np.asarray(img, dtype=float)

    # Normaliza para 0-16
    arr = 16.0 * (arr / 255.0)

    # Inverte caso o fundo esteja claro
    if arr.mean() > 8.0:
        arr = 16.0 - arr

    # Achata a imagem 8x8 para vetor de 64 posições
    X.append(arr.reshape(-1))
    y.append(rotulo)


# Converte para NumPy
X = np.asarray(X)
y = np.asarray(y)

print(f"Total de imagens encontradas: {len(X)}")
print(f"Total de imagens ignoradas: {len(ignorados)}")

# Mostra quantidade por classe
print("\nQuantidade por dígito:")
for digito in range(10):
    quantidade = np.sum(y == digito)
    print(f"  {digito}: {quantidade}")


if len(X) < 20:
    print("\nPoucas imagens para treino.")
    print("Adicione mais imagens na pasta 'dataset'.")

else:

    # Divide entre treinamento e teste
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    # Cria o KNN
    modelo = KNeighborsClassifier(n_neighbors=3)

    # Treina
    modelo.fit(X_train, y_train)

    # Faz previsões
    y_pred = modelo.predict(X_test)

    # Avaliação
    acc = accuracy_score(y_test, y_pred)

    print(f"\nAcurácia no conjunto de teste: {acc:.2%}")

    print("\nRelatório de classificação:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    if ignorados:
        print("\nArquivos ignorados por nome sem rótulo:")
        print(ignorados[:10])

Total de imagens encontradas: 1000
Total de imagens ignoradas: 0

Quantidade por dígito:
  0: 100
  1: 100
  2: 100
  3: 100
  4: 100
  5: 100
  6: 100
  7: 100
  8: 100
  9: 100

Acurácia no conjunto de teste: 87.20%

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        25
           1       0.70      0.84      0.76        25
           2       0.85      0.92      0.88        25
           3       0.86      0.76      0.81        25
           4       0.78      0.84      0.81        25
           5       0.89      0.68      0.77        25
           6       0.96      0.88      0.92        25
           7       0.95      0.80      0.87        25
           8       1.00      1.00      1.00        25
           9       0.83      1.00      0.91        25

    accuracy                           0.87       250
   macro avg       0.88      0.87      0.87       250
weighted avg       0.88      0.87      0.87      